# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 2: *Weather Data Processing*
##### Version Number: 2.0
---
### Contents  
> 1. *Data Loading and Exploration* 
> 2. *Data Cleaning*
> 3. *Data Merging*
> 4. *Add Supplementary Data*
> 5. *Export File*
---
### Notes

- Clean and process daily weather readings from **[California CIMIS irrigation stations](https://cimis.water.ca.gov/)** (2018-01-01 to 2024-12-31). Integrate wildfire impact data from the same period as damage cost in dollar amount.
---
### Inputs
- Daily Weather Readings - `CIMISdata.csv` 
- Weather Station Data - `i04_CIMIS_Weather_Stations.csv`
- Wildfire Damage Data - `clean_fires.csv` (cleaned in module 1)
- California Mesh Sampling Grid - `sampling_points.csv` (built in appendix A)
- 2020 US Census Data - `pop_data`, `mean_income` 
---
### Outputs  
- `model_fire_pop_income.csv` Cleaned weather dataset merged with fire damage severity, population and mean income data
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Convenience function to impute a column with the median
from src.data_utils import impute_median_data

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

# function to identify the number of missing dates in the dataset
from src.data_utils import identify_missing_station_dates

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

# Preprocessing and modeling utilities
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## Global Constants

In [3]:
# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

---

## 1. Data Loading and Exploration

### 1.1 CIMIS Historical Weather Data - `CIMISdata.csv`

This dataset contains daily weather measurements from CIMIS stations across California. Variables include temperature, precipitation, humidity, solar radiation, and wind. 
- Date Range = 01/01/2018 to 01/25/2025

Data sourced directly from:
[California CIMIS Data](https://cimis.water.ca.gov/WSNReportCriteria.aspx)

In [4]:
weather = pd.read_csv("../data/raw/C_data.csv")
weather.loc[:, 'Date'] = pd.to_datetime(weather['Date'], format='mixed').dt.date
basic_explore(weather)
weather.head(3)

Rows:  351978  Columns:  22
Duplicates  0
Total NA values:  73723  of  7743516 datapoints


,OBJECTID,Stn Id,Stn Name,CIMIS Region,Date,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Max Air Temp (F),...,Max Rel Hum (%),Min Rel Hum (%),Avg Rel Hum (%),Dew Point (F),Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F),County,Latitude,Longitude
0,1,2,FivePoints,San Joaquin Valley,2018-01-01,0.06,0.0,219.0,7.3,63.4,...,82.0,46.0,65.0,36.6,3.3,78.3,51.1,Fresno,36.336222,-120.112906
1,2,2,FivePoints,San Joaquin Valley,2018-01-02,0.04,0.0,127.0,7.4,59.8,...,80.0,52.0,67.0,36.7,3.1,74.5,51.3,Fresno,36.336222,-120.112906
2,3,2,FivePoints,San Joaquin Valley,2018-01-03,0.04,0.0,125.0,8.4,61.1,...,79.0,49.0,68.0,39.9,4.5,107.5,51.3,Fresno,36.336222,-120.112906


In [ ]:
weather_station = weather.drop_duplicates()
weather_station.duplicated().sum()

0

**initial observations:**
- The dataset contains `351,978` rows and `18` columns.
- `Date` column is currently a string object, conversion to date format is neccessary
- various columns contain `NULL` values

### 1.2 Wildfire Damage Data - `clean_fires.csv`

Provides destruction information from historical recorded wildfires that are used to construct the classification target (low, medium, high) for model training and evaluation. Relevant variables include:

- `Fire_Latitude` - Latitude of fire
- `Fire_Longitude` - Longitude of fire
- `Fire Name` - Fire Name
- `TotalEstimatedCost` - Estimated dollar amount of damage caused

Source: [CAL FIRE – Fire Incidents](https://www.fire.ca.gov/)  
This data was extracted and structured from publicly available incident records for analytical use. See Module 1 for details.

In [7]:
fire_damage = pd.read_csv("../data/raw/damage/clean_fires.csv")
basic_explore(fire_damage)

Rows:  269  Columns:  7
Duplicates  0
Total NA values:  0  of  1883 datapoints


In [ ]:
## Load Geometry for future use
fire_damage['geometry'] = [Point(xy) for xy in zip(fire_damage['Fire_Longitude'], fire_damage['Fire_Latitude'])]
fire_damage_gdf = gpd.GeoDataFrame(fire_damage, geometry='geometry', crs="EPSG:4326")

In [ ]:
fire_damage_gdf.sample(3)

,Unnamed: 0,FireOutDateTime,IncidentName,Fire_Latitude,Fire_Longitude,EstimatedFinalCost,Date
177,45876,2019-07-17,LUMGREY,41.866300,-122.743700,1257745.0,2019-06-18
82,40666,2020-07-05,AURORA,38.229940,-118.980000,150000.0,2020-06-26
247,70150,2024-11-14,MILL,39.740583,-120.522799,40000000.0,2024-07-22


In [ ]:
fire_damage_gdfinfo()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 269 entries, 0 to 268
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          269 non-null    int64  
 1   FireOutDateTime     269 non-null    object 
 2   IncidentName        269 non-null    object 
 3   Fire_Latitude       269 non-null    float64
 4   Fire_Longitude      269 non-null    float64
 5   EstimatedFinalCost  269 non-null    float64
 6   Date                269 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 14.8+ KB


In [ ]:
after_total = fire_damage_gdf['EstimatedFinalCost'].values.sum()
print(f"Total Estimated Value: ${after_total:,.0f}")

Total Estimated Value: $6,985,379,871


**initial observations:**
- The dataset contains `172` rows (wildfire incidents) and `10` columns.
- Records without finish times were imputed with the mean number duration of the rest of the dataset

### 1.3 Load Mesh Network

<img src="../data/maps/mesh.jpg" width="600">

In [ ]:
## Load file
samples = pd.read_csv("../data/raw/sampling_points.csv")

# Load geometry
samples['geometry'] = [Point(xy) for xy in zip(samples['Sample_Longitude'], samples['Sample_Latitude'])]
samples_gdf = gpd.GeoDataFrame(samples, geometry='geometry', crs="EPSG:4326")

## generate unique identifier
samples_gdf['Sample_ID'] = range(1, len(samples_gdf) + 1)

## fix error from ArcGIS
samples_gdf['County'] = samples_gdf['County'].replace('South Coast Valleys', 'San Diego')

samples_gdf


### Expand Sampling points
- Add each date in range of weather station data `01/01/2018` to `01/23/2025`
- Preparation for merge of weather readings and sampling points

In [ ]:
# Define date range
dates = pd.date_range(start="2018-01-01", end="2025-01-23", freq="D")

# make a safe copy
points_df = samples_gdf.copy()  

# Add a key to merge on
points_df['key'] = 1
dates_df = pd.DataFrame({'Date': dates})
dates_df['key'] = 1

# Merge dates dataframe with sampling points on generated key
daily_samples_gdf = points_df.merge(dates_df, on='key').drop('key', axis=1)

daily_samples_gdf.isna().sum()

---

## 2. Data Cleaning

### 2.1. Data Imputation

Display percentage of the amount of NA values in each column

In [13]:
NA_values = weather_station.isna().mean().round(4) * 100
NA_values = NA_values[NA_values > 0]
NA_values = NA_values[NA_values > 0].map(lambda x: f"{x:.2f}%")
NA_values

ETo (in)                5.91%
Precip (in)             0.75%
Sol Rad (Ly/day)        0.73%
Avg Vap Pres (mBars)    0.87%
Max Air Temp (F)        1.33%
Min Air Temp (F)        1.02%
Avg Air Temp (F)        0.96%
Max Rel Hum (%)         0.92%
Min Rel Hum (%)         0.92%
Avg Rel Hum (%)         1.67%
Dew Point (F)           1.67%
Avg Wind Speed (mph)    0.75%
Wind Run (miles)        0.75%
Avg Soil Temp (F)       2.70%
dtype: object

*impute_median_data* - code from src.data_utils that fills all NA values with the median of each column.

In [14]:
# Fill missing numeric weather values in the data dataframe using the median of each column.
weather_station = impute_median_data(weather_station)

In [16]:
weather_station.isna().sum()

OBJECTID                0
Stn Id                  0
Stn Name                0
CIMIS Region            0
Date                    0
ETo (in)                0
Precip (in)             0
Sol Rad (Ly/day)        0
Avg Vap Pres (mBars)    0
Max Air Temp (F)        0
Min Air Temp (F)        0
Avg Air Temp (F)        0
Max Rel Hum (%)         0
Min Rel Hum (%)         0
Avg Rel Hum (%)         0
Dew Point (F)           0
Avg Wind Speed (mph)    0
Wind Run (miles)        0
Avg Soil Temp (F)       0
County                  0
Latitude                0
Longitude               0
dtype: int64

### 2.2 Identify Missing Dates
Some weather stations have gaps in daily records due to maintenance or equipment failures. Loop through all counties and compare the actual recorded dates to the expected continuous date range to identify missing entries.

**Note:** These missing dates could cause some fires to not be matched. Effect on overall dataset is minimal but these dates could be imputed in future updates.

In [ ]:
missing_dates = identify_missing_station_dates(weather_station)
print("Number of missing entries: ", len(missing_dates))

In [ ]:
# Define date range
dates = pd.date_range(start="2018-01-01", end="2025-01-23", freq="D")

# make a safe copy
points_df = samples_gdf.copy()  

# Add a key to merge on
points_df['key'] = 1
dates_df = pd.DataFrame({'Date': dates})
dates_df['key'] = 1

# Merge dates dataframe with sampling points on generated key
daily_samples_gdf = points_df.merge(dates_df, on='key').drop('key', axis=1)

daily_samples_gdf.isna().sum()

---

## 3. Merging

### 3.1 Merge Sampling Points and Weather data

In [ ]:
## Load Weather Station geometry 
weather_station['geometry'] = [Point(xy) for xy in zip(weather_station['Longitude'], weather_station['Latitude'])]
stations_gdf = gpd.GeoDataFrame(weather_station, geometry='geometry', crs="EPSG:4326")

## rename for upcoming merge
stations_gdf = stations_gdf.rename(columns={'Stn Id': 'Stn_Id'})

# Convert date columns to datetime
daily_samples_gdf['Date'] = pd.to_datetime(daily_samples_gdf['Date'])
stations_gdf['Date'] = pd.to_datetime(stations_gdf['Date'])

# Merge on BOTH station and date
samples_daily = daily_samples_gdf.merge(
    stations_gdf,
    on=['Stn_Id', 'Date'],
    how='left'
)

## Reorganize so Sample ID and Date are first two columns
samples_daily.insert(0, 'Date', samples_daily.pop('Date'))
samples_daily.insert(0, 'Sample_ID', samples_daily.pop('Sample_ID'))

In [35]:
samples_daily = impute_median_data(samples_daily)

In [36]:
samples_daily.isna().sum()

Sample_ID                   0
Date                        0
Join_Count                  0
TARGET_FID                  0
Sample_Longitude            0
Sample_Latitude             0
Sample_Elevation            0
Region_ID                   0
Region_Name                 0
Stn_Id                      0
Stn_Name                    0
County_x                    0
geometry_x                  0
OBJECTID                    0
Stn Name                47037
CIMIS Region            47037
ETo (in)                    0
Precip (in)                 0
Sol Rad (Ly/day)            0
Avg Vap Pres (mBars)        0
Max Air Temp (F)            0
Min Air Temp (F)            0
Avg Air Temp (F)            0
Max Rel Hum (%)             0
Min Rel Hum (%)             0
Avg Rel Hum (%)             0
Dew Point (F)               0
Avg Wind Speed (mph)        0
Wind Run (miles)            0
Avg Soil Temp (F)           0
County_y                47037
Latitude                    0
Longitude                   0
geometry_y

#### Calculate Season for each entry
`Winter` = 0, `Spring` = 1, `Summer` = 2, `Fall` = 3

In [ ]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 0
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3

## Apply function
samples_daily['Season'] = samples_daily['Date'].apply(get_season)

### 3.2 Spatial Join of Nearest Sampling Locations with Fire Damage Datasets

To prepare for feature engineering, we spatially join each fire location to its nearest sampling point. This enables us to associate daily environmental conditions with each fire based on geographic proximity, rather than relying solely on county or administrative boundaries.

In [ ]:
samples_daily_gdf = gpd.GeoDataFrame(samples_daily, geometry='geometry_x', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
samples_projected = samples_daily_gdf.to_crs(3310)
fire_damage_projected = fire_damage_gdf.to_crs(3310)

In [ ]:
## Buffer distance in meters, Sampling points are located approximately 46000 meters apart
buffer_dist = 200000

In [ ]:
## Initialize variables to track total damage and number of fires associated with each point
samples_projected['fire_count'] = 0
fire_damage_projected['total_fire_damage'] = 0.0

#### Main loop of Spatial Join algorithm
- Loop through all dates in fire damage dataset
- Save all fires associated with the current date
    - If no fires, move to next date
- Load sampling points associated with current date
- Create a buffer, sized as defined earlier, around each fire for current date
- Intersection spatial join of sampling points and buffered fires
- Aggregate in case of multiple fires in a buffered range
    - Total estimated cost is summed for all fires in range
- Assign samples to dataframe

In [ ]:
for dt in fire_damage_projected['Date'].unique():
    
    # Fires on this date
    fires_today = fire_damage_projected[fire_damage_projected['Date'] == dt]
    if fires_today.empty:
        continue

    # Samples on this date
    samples_today = samples_projected[samples_projected['Date'] == dt]
    if samples_today.empty:
        continue

    # Create buffers around each fire
    fires_buffered = fires_today.copy()
    fires_buffered['geometry'] = fires_buffered.buffer(buffer_dist)

    # Spatial join: find samples within fire buffers
    joined = gpd.sjoin(samples_today, fires_buffered, how='left', predicate='intersects')

    # Aggregate counts and total damage per sample
    grouped = joined.groupby('Sample_ID').agg(
        fire_count=('EstimatedFinalCost', 'count'),
        total_fire_damage=('EstimatedFinalCost', 'sum')
    ).fillna(0)

    # Assign values back to main dataframe
    samples_projected.loc[samples_projected['Date'] == dt, 'fire_count'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'Sample_ID'].map(grouped['fire_count']).fillna(0)
    samples_projected.loc[samples_projected['Date'] == dt, 'total_fire_damage'] = \
        samples_projected.loc[samples_projected['Date'] == dt, 'Sample_ID'].map(grouped['total_fire_damage']).fillna(0)


In [47]:
samples_projected['total_fire_damage'] = samples_projected['total_fire_damage'].fillna(0)

In [48]:
samples_projected

,Sample_ID,Date,Join_Count,TARGET_FID,Sample_Longitude,Sample_Latitude,Sample_Elevation,Region_ID,Region_Name,Stn_Id,...,Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F),County_y,Latitude,Longitude,geometry_y,Season,fire_count,total_fire_damage
0,1,2018-01-01,0,1,-117.232017,32.778837,36.609302,7,Marine Region,173,...,3.1,74.6,56.8,San Diego,32.901867,-117.250458,POINT (-117.25046 32.90187),0,0,0.0
1,1,2018-01-02,0,1,-117.232017,32.778837,36.609302,7,Marine Region,173,...,3.5,84.0,57.8,San Diego,32.901867,-117.250458,POINT (-117.25046 32.90187),0,0,0.0
2,1,2018-01-03,0,1,-117.232017,32.778837,36.609302,7,Marine Region,173,...,2.9,70.3,58.3,San Diego,32.901867,-117.250458,POINT (-117.25046 32.90187),0,0,0.0
3,1,2018-01-04,0,1,-117.232017,32.778837,36.609302,7,Marine Region,173,...,3.6,85.8,59.0,San Diego,32.901867,-117.250458,POINT (-117.25046 32.90187),0,0,0.0
4,1,2018-01-05,0,1,-117.232017,32.778837,36.609302,7,Marine Region,173,...,4.5,108.0,58.8,San Diego,32.901867,-117.250458,POINT (-117.25046 32.90187),0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,173,2025-01-19,1,173,-120.232017,41.778837,1618.552856,1,Northern Region,90,...,3.7,88.0,33.7,Modoc,41.438214,-120.480308,POINT (-120.48031 41.43821),0,0,0.0
447776,173,2025-01-20,1,173,-120.232017,41.778837,1618.552856,1,Northern Region,90,...,3.6,85.5,33.6,Modoc,41.438214,-120.480308,POINT (-120.48031 41.43821),0,0,0.0
447777,173,2025-01-21,1,173,-120.232017,41.778837,1618.552856,1,Northern Region,90,...,3.0,71.0,33.4,Modoc,41.438214,-120.480308,POINT (-120.48031 41.43821),0,0,0.0
447778,173,2025-01-22,1,173,-120.232017,41.778837,1618.552856,1,Northern Region,90,...,3.1,73.7,33.2,Modoc,41.438214,-120.480308,POINT (-120.48031 41.43821),0,0,0.0


#### Clean resulting dataframe

In [52]:
samples_projected.columns

Index(['Sample_ID', 'Date', 'Join_Count', 'TARGET_FID', 'Sample_Longitude',
       'Sample_Latitude', 'Sample_Elevation', 'Region_ID', 'Region_Name',
       'Stn_Id', 'Stn_Name', 'County_x', 'geometry_x', 'OBJECTID', 'Stn Name',
       'CIMIS Region', 'ETo (in)', 'Precip (in)', 'Sol Rad (Ly/day)',
       'Avg Vap Pres (mBars)', 'Max Air Temp (F)', 'Min Air Temp (F)',
       'Avg Air Temp (F)', 'Max Rel Hum (%)', 'Min Rel Hum (%)',
       'Avg Rel Hum (%)', 'Dew Point (F)', 'Avg Wind Speed (mph)',
       'Wind Run (miles)', 'Avg Soil Temp (F)', 'County_y', 'Latitude',
       'Longitude', 'geometry_y', 'Season', 'fire_count', 'total_fire_damage'],
      dtype='object')

In [53]:
drop = ['TARGET_FID', 'Join_Count','County_y', 'geometry_y', 'OBJECTID', 'Latitude',
       'Longitude', 'geometry_y']
samples_projected = samples_projected.drop(columns=drop)
samples_projected.rename(columns={'County_x': 'County'}, inplace=True)

***Note*** NA values are in fields that are unused in future operations.

In [54]:
samples_projected.isna().sum()

Sample_ID                   0
Date                        0
Sample_Longitude            0
Sample_Latitude             0
Sample_Elevation            0
Region_ID                   0
Region_Name                 0
Stn_Id                      0
Stn_Name                    0
County                      0
geometry_x                  0
Stn Name                47037
CIMIS Region            47037
ETo (in)                    0
Precip (in)                 0
Sol Rad (Ly/day)            0
Avg Vap Pres (mBars)        0
Max Air Temp (F)            0
Min Air Temp (F)            0
Avg Air Temp (F)            0
Max Rel Hum (%)             0
Min Rel Hum (%)             0
Avg Rel Hum (%)             0
Dew Point (F)               0
Avg Wind Speed (mph)        0
Wind Run (miles)            0
Avg Soil Temp (F)           0
Season                      0
fire_count                  0
total_fire_damage           0
dtype: int64

---

## 4. Suplementary Data

### 4.1 Merge population density data

Add Population Data obtained from 2020 US Census Data.\
- `Population Density`
- `Total Population` in a County

In [55]:
pop_density = pd.read_csv("../data/raw/pop.csv")
basic_explore(pop_density)

Rows:  58  Columns:  8
Duplicates  0
Total NA values:  0  of  464 datapoints


Basic formatting of column names and cases for merging

In [56]:
pop_density.rename(columns={'POPESTIMATE': 'Total Population'}, inplace=True)
pop_keep = pop_density.drop(columns = ['Unnamed: 0','County','YEAR','Size (square miles)','class'])
pop_keep.rename(columns={'CTYNAME': 'County'}, inplace=True)
#pop_keep['County'] = pop_keep['County'].str.upper()
pop_keep.head(2)

,County,Total Population,density
0,Alameda,1622188,2195.052908
1,Alpine,1141,1.545379


Merge in population data to main dataset and case study datasets.

In [60]:
fires_pop = samples_projected.merge(
    pop_keep,
    left_on='County', right_on='County',
    how='left', indicator=True
)

In [61]:
print("Main Weather Data:")
post_merge_check(fires_pop,samples_projected)
fires_pop = fires_pop.drop(columns=['_merge'])

Main Weather Data:
Premerged shape:  (447780, 30)
Merged shape:  (447780, 33)
Duplicates before merge:  1440
Duplicates after merge:  1440
NA values before merge:  94074
NA values after merge:  94074


### 4.2 Merge Mean Income Data

Add mean income data per county obtained from 2020 US Census Data. This serves as a general proxy for firefighting resources.

In [63]:
mean_income = pd.read_csv("../data/raw/mean_income_by_county.csv")
basic_explore(mean_income)

Rows:  58  Columns:  2
Duplicates  0
Total NA values:  0  of  116 datapoints


format data to match case and adjust `Mean Income` column to be numerical

In [64]:
#mean_income['County'] = mean_income['County'].str.upper()
mean_income['Mean Income'] = (
    mean_income['Mean Income']
    .str.replace(',', '', regex=False)  # Remove commas
    .astype(int)                        # Convert to integer
)

mean_income.head(2)

,County,Mean Income
0,Alameda,138489
1,Alpine,107737


Merge mean income data into main dataset and case study data

In [65]:
fire_pop_income = fires_pop.merge(
    mean_income,
    left_on='County', right_on='County',
    how='left', indicator=True
)

In [66]:
print("Main Weather Data:")
post_merge_check(fire_pop_income,fires_pop)

Main Weather Data:
Premerged shape:  (447780, 32)
Merged shape:  (447780, 34)
Duplicates before merge:  1440
Duplicates after merge:  1440
NA values before merge:  94074
NA values after merge:  94074


In [67]:
#fire_pop_income = fire_pop_income.drop(columns='_merge')
fire_pop_income.head(2)

,Sample_ID,Date,Sample_Longitude,Sample_Latitude,Sample_Elevation,Region_ID,Region_Name,Stn_Id,Stn_Name,County,...,Avg Wind Speed (mph),Wind Run (miles),Avg Soil Temp (F),Season,fire_count,total_fire_damage,Total Population,density,Mean Income,_merge
0,1,2018-01-01,-117.232017,32.778837,36.609302,7,Marine Region,173,Torrey Pines,San Diego,...,3.1,74.6,56.8,0,0,0.0,3269973,777.337917,111241,both
1,1,2018-01-02,-117.232017,32.778837,36.609302,7,Marine Region,173,Torrey Pines,San Diego,...,3.5,84.0,57.8,0,0,0.0,3269973,777.337917,111241,both


---

## 5. Build Final Target

Create a categorical target that represents the risk levels of damage caused.\
\
`Low` Risk = 0, No fires started on the day\
`Medium` Risk = 1, Fires present but limited property damage done, under 1 million dollars total\
`High` Risk = 2, Fires causing over 1 million dollars in damage 

In [ ]:
def fire_risk_category(row):
    # Low risk: no fires
    if row['fire_count'] == 0:
        return 0

    # Moderate risk: at least one fire AND damage under 1 million
    elif row['total_fire_damage'] < 1e6:
        return 1

    # High risk: all other cases
    else:
        return 2

## Apply function
fire_pop_income['Target'] = fire_pop_income.apply(fire_risk_category, axis=1)

## Display resulting risk category assignments
fire_pop_income['Target'].value_counts()

#### Clean Dataframe

In [ ]:
## sort dataframe by date
fire_pop_income = fire_pop_income.sort_values(by='Date')

## reset index
fire_pop_income = fire_pop_income.reset_index(drop=True)

Index(['Sample_ID', 'Date', 'Sample_Longitude', 'Sample_Latitude',
       'Sample_Elevation', 'Region_ID', 'Region_Name', 'Stn_Id', 'Stn_Name',
       'County', 'geometry_x', 'Stn Name', 'CIMIS Region', 'ETo (in)',
       'Precip (in)', 'Sol Rad (Ly/day)', 'Avg Vap Pres (mBars)',
       'Max Air Temp (F)', 'Min Air Temp (F)', 'Avg Air Temp (F)',
       'Max Rel Hum (%)', 'Min Rel Hum (%)', 'Avg Rel Hum (%)',
       'Dew Point (F)', 'Avg Wind Speed (mph)', 'Wind Run (miles)',
       'Avg Soil Temp (F)', 'Season', 'fire_count', 'total_fire_damage',
       'Total Population', 'density', 'Mean Income', '_merge', 'Target'],
      dtype='object')

In [82]:
fire_pop_income.columns

Index(['Sample_ID', 'Date', 'Sample_Longitude', 'Sample_Latitude',
       'Sample_Elevation', 'Region_ID', 'Region_Name', 'Stn_Id', 'Stn_Name',
       'County', 'geometry_x', 'Stn Name', 'CIMIS Region', 'ETo (in)',
       'Precip (in)', 'Sol Rad (Ly/day)', 'Avg Vap Pres (mBars)',
       'Max Air Temp (F)', 'Min Air Temp (F)', 'Avg Air Temp (F)',
       'Max Rel Hum (%)', 'Min Rel Hum (%)', 'Avg Rel Hum (%)',
       'Dew Point (F)', 'Avg Wind Speed (mph)', 'Wind Run (miles)',
       'Avg Soil Temp (F)', 'Season', 'fire_count', 'total_fire_damage',
       'Total Population', 'density', 'Mean Income', '_merge', 'Target'],
      dtype='object')

In [ ]:
drop_cols =['Sample_Longitude', 'Sample_Latitude', 'Region_Name', 'Stn_Id', 'Stn_Name',
       'County', 'geometry_x', 'Stn Name', 'CIMIS Region', 'fire_count', 'total_fire_damage',
        '_merge']

## drop unneeded columns
model_fire_pop_income = fire_pop_income.drop(columns=drop_cols)

## Save some additional useful columns for further analysis
keep_columns = ['Sample_ID','Date','Region_ID'] 

#  build list of columns to save in details
drop_cols_with_id =  keep_columns + drop_cols

details = fire_pop_income[drop_cols_with_id].copy()

## (Optional) Limit minority class to 100,000 samples to save processing power

In [ ]:
# Separate by target class
target_0 = fire_pop_income[fire_pop_income['Target'] == 0]
target_1_2 = fire_pop_income[fire_pop_income['Target'].isin([1, 2])]

# Randomly sample 5000 rows from class 0
target_0_sampled = target_0.sample(n=100000, random_state=42)

# Combine with all class 1 and 2 rows
fire_pop_income_balanced = pd.concat([target_0_sampled, target_1_2], ignore_index=True)

print(fire_pop_income_balanced['Target'].value_counts())

---

## 5. Export File

In [ ]:
model_fire_pop_income.to_csv('../data/processed/model_fire_pop_income.csv',index=False)
details.to_csv('../data/processed/details.csv',index=False)
print("All datasets saved successfully to ../data/processed/")